# Activation Space vs. Behavior Space: A Goodfire Demo

> **Educational miniature** of the Goodfire Manifold Steering approach (arXiv:2605.05115).
> Not a full paper reproduction — an illustration of the core geometric claim on GPT-2.

**Core claim (Goodfire):** The geometry of how a model *internally represents* concepts
(activation space) approximately mirrors the geometry of how it *behaves* on those concepts
(behavior space). This approximate match is called an **isometry**.

**Model:** GPT-2 (local, ~500 MB, no API key)
**Concepts:** Days of the week — an ordered cyclic set with clear sequential structure in
GPT-2's training data. This matches the example used in the Goodfire paper.

In [ ]:
import warnings
from itertools import combinations
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from scipy.stats import pearsonr
from scipy.spatial.distance import euclidean
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100

from prompts import CONCEPT_PROMPTS, DAYS

print(f"Concepts: {DAYS}")
print(f"Total prompts: {sum(len(v) for v in CONCEPT_PROMPTS.values())}")

## Section 1: What Is Approximate Isometry? (Synthetic)

Before loading any model, we understand "approximate isometry" geometrically.

Synthetic activation and behavior centroids are arranged in a cycle (like days of the week),
related by a noisy linear map. The isometry check: do their pairwise distances correlate?

If the scatter of (d_activation, d_behavior) for all concept pairs falls near a line,
the two spaces are approximately isometric — same shape, possibly different scale/rotation.

In [ ]:
np.random.seed(42)
N = 7
short_labels = [d[:3] for d in DAYS]
angles    = np.linspace(0, 2 * np.pi, N, endpoint=False)
act_synth = np.column_stack([np.cos(angles), np.sin(angles)]) * 2.0

A         = np.array([[0.8, -0.5], [0.3, 1.2]])
beh_synth = (A @ act_synth.T).T + np.random.randn(N, 2) * 0.18

colors_synth = [plt.cm.tab10(i / N) for i in range(N)]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, data, title in [
    (axes[0], act_synth, "Synthetic Activation Space"),
    (axes[1], beh_synth, "Synthetic Behavior Space"),
]:
    for i in range(N):
        ax.scatter(*data[i], color=colors_synth[i], s=130, zorder=3)
        ax.annotate(short_labels[i], data[i] + np.array([0.08, 0.08]), fontsize=10)
    for i in range(N):
        j = (i + 1) % N
        ax.plot([data[i,0], data[j,0]], [data[i,1], data[j,1]],
                color="gray", alpha=0.4, lw=1.5)
    ax.set_title(title, fontsize=12)
    ax.set_aspect("equal")
    ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2")

plt.suptitle("Same cyclic concept geometry — two different spaces", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
pair_idxs   = list(combinations(range(N), 2))
d_act_synth = np.array([np.linalg.norm(act_synth[i]-act_synth[j]) for i,j in pair_idxs])
d_beh_synth = np.array([np.linalg.norm(beh_synth[i]-beh_synth[j]) for i,j in pair_idxs])
adj_synth   = np.array([abs(i-j)==1 or abs(i-j)==N-1 for i,j in pair_idxs])

r_synth, _ = pearsonr(d_act_synth, d_beh_synth)
m_synth     = np.polyfit(d_act_synth, d_beh_synth, 1)
x_line      = np.linspace(d_act_synth.min(), d_act_synth.max(), 100)

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(d_act_synth[adj_synth],  d_beh_synth[adj_synth],
           color="steelblue", s=80, alpha=0.9, label="Adjacent pairs")
ax.scatter(d_act_synth[~adj_synth], d_beh_synth[~adj_synth],
           color="coral",     s=60, alpha=0.7, label="Non-adjacent pairs")
ax.plot(x_line, np.polyval(m_synth, x_line), "k--", lw=1.5,
        label=f"fit (slope={m_synth[0]:.2f})")
ax.set_xlabel("d_activation (synthetic)", fontsize=11)
ax.set_ylabel("d_behavior (synthetic)",   fontsize=11)
ax.set_title(f"Pairwise Distance Scatter\nPearson r = {r_synth:.3f}", fontsize=12)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f"Synthetic Pearson r = {r_synth:.4f}  (A is non-isometric + noise: approximate isometry by construction)")

## Section 2: Activation Space (GPT-2 Hidden States)

For each prompt we extract GPT-2's final-layer hidden state at the last token position
(768-dim vector) — the model's internal representation just before deciding what to output.

Pipeline: tokenize -> run with output_hidden_states=True -> hidden_states[-1][0, -1, :]
-> 768D vector -> 105 vectors total -> PCA (10D for distances, 2D for visualization) -> centroids.

In [ ]:
print("Loading GPT-2 (downloads ~500MB on first run, then cached)...")
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model     = GPT2LMHeadModel.from_pretrained("gpt2")
model.train(False)   # set to inference mode
print(f"Model loaded. Hidden dim: {model.config.n_embd}, Vocab: {model.config.vocab_size}")

In [ ]:
def get_hidden_state(prompt: str) -> np.ndarray:
    """Final-layer hidden state at last token position. Shape: (768,)"""
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
    # hidden_states is a tuple: (embedding + one per transformer layer)
    # Index -1 is the final transformer block output; shape (1, seq_len, 768)
    return outputs.hidden_states[-1][0, -1, :].numpy()

print("Extracting hidden states (105 prompts)...")
hidden_vectors, hidden_labels = [], []
for day in DAYS:
    for prompt in CONCEPT_PROMPTS[day]:
        hidden_vectors.append(get_hidden_state(prompt))
        hidden_labels.append(day)

hidden_vectors = np.array(hidden_vectors)   # (105, 768)
hidden_labels  = np.array(hidden_labels)
print(f"Collected: {hidden_vectors.shape}")

pca_act_10d = PCA(n_components=10, random_state=42)
pca_act_2d  = PCA(n_components=2,  random_state=42)
act_10d = pca_act_10d.fit_transform(hidden_vectors)
act_2d  = pca_act_2d.fit_transform(hidden_vectors)

act_centroids_10d = {day: act_10d[hidden_labels == day].mean(axis=0) for day in DAYS}
act_centroids_2d  = {day: act_2d[hidden_labels == day].mean(axis=0)  for day in DAYS}

print(f"PCA done. Explained variance (10 PCs): {pca_act_10d.explained_variance_ratio_.sum():.1%}")

In [ ]:
colors_map = {day: plt.cm.tab10(i / len(DAYS)) for i, day in enumerate(DAYS)}
short      = {d: d[:3] for d in DAYS}

fig, ax = plt.subplots(figsize=(7, 6))
for day in DAYS:
    mask = hidden_labels == day
    ax.scatter(act_2d[mask, 0], act_2d[mask, 1],
               color=colors_map[day], alpha=0.25, s=30)
for day in DAYS:
    ax.scatter(*act_centroids_2d[day], color=colors_map[day], s=180, zorder=4,
               edgecolors="white", linewidths=1.5, label=short[day])
for i in range(len(DAYS)):
    j = (i + 1) % len(DAYS)
    c1, c2 = act_centroids_2d[DAYS[i]], act_centroids_2d[DAYS[j]]
    ax.plot([c1[0], c2[0]], [c1[1], c2[1]], color="gray", lw=1.2, alpha=0.5)

ax.set_title("Activation Space — GPT-2 final hidden states (PCA 2D)\nCentroids connected in day-of-week order", fontsize=11)
ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2")
ax.legend(title="Day", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)
plt.tight_layout()
plt.show()

## Section 3: Behavior Space (Output Probability Distributions)

For each prompt we take GPT-2's output probability distribution over 50,257 tokens.

Why sqrt(p) — the Hellinger embedding?
The probability simplex is not Euclidean. The square-root map sends each distribution to the
positive orthant of the unit sphere in R^50257, where Euclidean distance = Hellinger distance:

    ||sqrt(p) - sqrt(q)||_2  =  sqrt(2) * H(p, q)

This makes PCA, centroids, and distance metrics geometrically valid.
PCA on raw probabilities has no geometric justification; PCA on sqrt(p) does.

In [ ]:
def get_probs(prompt: str) -> np.ndarray:
    """Softmax output probability distribution. Shape: (50257,)"""
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
    return torch.softmax(outputs.logits[0, -1, :], dim=-1).numpy()

print("Extracting output probabilities (105 prompts)...")
prob_vectors, prob_labels = [], []
for day in DAYS:
    for prompt in CONCEPT_PROMPTS[day]:
        prob_vectors.append(get_probs(prompt))
        prob_labels.append(day)

prob_vectors = np.array(prob_vectors)   # (105, 50257)
prob_labels  = np.array(prob_labels)
hell_vectors = np.sqrt(prob_vectors)    # Hellinger embedding: maps to unit sphere positive orthant

print(f"Prob vectors:      {prob_vectors.shape}")
print(f"Hellinger vectors: {hell_vectors.shape}")
print(f"Norm check (should be ~1.0): {np.linalg.norm(hell_vectors[0]):.6f}")

In [ ]:
pca_beh_10d = PCA(n_components=10, random_state=42)
pca_beh_2d  = PCA(n_components=2,  random_state=42)
beh_10d = pca_beh_10d.fit_transform(hell_vectors)
beh_2d  = pca_beh_2d.fit_transform(hell_vectors)

beh_centroids_10d = {day: beh_10d[prob_labels == day].mean(axis=0) for day in DAYS}
beh_centroids_2d  = {day: beh_2d[prob_labels == day].mean(axis=0)  for day in DAYS}

print(f"Explained variance (10 PCs, behavior): {pca_beh_10d.explained_variance_ratio_.sum():.1%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
specs = [
    (axes[0], act_centroids_2d, act_2d, hidden_labels, "Activation Space\n(hidden states, PCA 2D)"),
    (axes[1], beh_centroids_2d, beh_2d, prob_labels,   "Behavior Space\n(sqrt-p Hellinger, PCA 2D)"),
]
for ax, centroids_2d, all_2d, labels_arr, title in specs:
    for day in DAYS:
        mask = labels_arr == day
        ax.scatter(all_2d[mask, 0], all_2d[mask, 1],
                   color=colors_map[day], alpha=0.2, s=25)
    for day in DAYS:
        ax.scatter(*centroids_2d[day], color=colors_map[day], s=180, zorder=4,
                   edgecolors="white", linewidths=1.5, label=short[day])
    for i in range(len(DAYS)):
        j = (i + 1) % len(DAYS)
        c1, c2 = centroids_2d[DAYS[i]], centroids_2d[DAYS[j]]
        ax.plot([c1[0], c2[0]], [c1[1], c2[1]], color="gray", lw=1.2, alpha=0.5)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2")
    ax.legend(title="Day", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)

plt.suptitle("Activation Space vs. Behavior Space — same concepts, different geometry", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
pairs       = list(combinations(DAYS, 2))
d_act_real  = np.array([euclidean(act_centroids_10d[d1], act_centroids_10d[d2]) for d1,d2 in pairs])
d_beh_real  = np.array([euclidean(beh_centroids_10d[d1], beh_centroids_10d[d2]) for d1,d2 in pairs])

def cyclic_gap(d1, d2):
    i, j = DAYS.index(d1), DAYS.index(d2)
    return min(abs(i - j), len(DAYS) - abs(i - j))

adj_flags_real = np.array([cyclic_gap(d1, d2) == 1 for d1, d2 in pairs])
stress_flags   = np.array([{d1, d2} == {"Monday", "Friday"} for d1, d2 in pairs])

r_real, p_val = pearsonr(d_act_real, d_beh_real)
m_real = np.polyfit(d_act_real, d_beh_real, 1)
x_line = np.linspace(d_act_real.min(), d_act_real.max(), 100)

fig, ax = plt.subplots(figsize=(7, 6))
mask_other = ~adj_flags_real & ~stress_flags
ax.scatter(d_act_real[mask_other],     d_beh_real[mask_other],
           color="coral",     s=60, alpha=0.7, label="Non-adjacent pairs")
ax.scatter(d_act_real[adj_flags_real], d_beh_real[adj_flags_real],
           color="steelblue", s=80, alpha=0.9, label="Adjacent pairs (cycle-local)")
ax.scatter(d_act_real[stress_flags],   d_beh_real[stress_flags],
           color="red", s=150, zorder=5, marker="*", label="Stress pair (Mon-Fri)")
ax.plot(x_line, np.polyval(m_real, x_line), "k--", lw=1.5,
        label=f"linear fit (slope={m_real[0]:.2f})")
ax.set_xlabel("d_activation (10D PCA)", fontsize=11)
ax.set_ylabel("d_behavior (10D sqrt-p PCA)", fontsize=11)
ax.set_title(f"Isometry Plot — Pairwise Distance Correlation\nPearson r = {r_real:.3f}  (p = {p_val:.4f})", fontsize=11)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

mon_tue_idx = next(i for i,(d1,d2) in enumerate(pairs) if {d1,d2} == {"Monday","Tuesday"})
mon_fri_idx = next(i for i,(d1,d2) in enumerate(pairs) if {d1,d2} == {"Monday","Friday"})
print(f"Pearson r = {r_real:.4f}")
print(f"Mon<->Tue  activation: {d_act_real[mon_tue_idx]:.3f},  behavior: {d_beh_real[mon_tue_idx]:.3f}")
print(f"Mon<->Fri  activation: {d_act_real[mon_fri_idx]:.3f},  behavior: {d_beh_real[mon_fri_idx]:.3f}")

## Section 4: Activation Steering Demo + Stress Test

Goodfire's manifold steering adds a direction vector to the model's hidden state.
If isometry holds: smooth move in activation space -> smooth move in behavior space.

Steering direction: Tuesday_centroid - Monday_centroid in 768D activation space.

We hook into GPT-2's final transformer block (.h[-1]) and add alpha * direction
before the unembedding step, observing how the output distribution shifts.

Stress test: Apply the same direction starting from a Friday prompt (cross-cycle boundary).
If isometry is local to the cycle, the Friday behavior trajectory may be less predictable.

Note: GPT-2 is not instruction-tuned, so refusal-style discontinuities will not appear.
We test geometric boundary effects (adjacent vs. cross-cycle) as a proxy for the "crack".

In [ ]:
def get_centroid_768d(day: str) -> np.ndarray:
    """Mean hidden state across all 15 prompts for a day. Shape: (768,)"""
    return hidden_vectors[hidden_labels == day].mean(axis=0)

print("Computing 768D centroids for Monday, Tuesday, Friday...")
mon_768 = get_centroid_768d("Monday")
tue_768 = get_centroid_768d("Tuesday")
fri_768 = get_centroid_768d("Friday")

direction      = tue_768 - mon_768
direction_norm = direction / np.linalg.norm(direction)

print(f"Mon centroid norm:       {np.linalg.norm(mon_768):.2f}")
print(f"Steering direction norm: {np.linalg.norm(direction):.4f}")
print(f"Unit direction norm:     {np.linalg.norm(direction_norm):.6f}  (should be 1.0)")

In [ ]:
def get_steered_probs(prompt: str, direction: np.ndarray, alpha: float) -> np.ndarray:
    """
    Run GPT-2 with (alpha * direction) added to the final transformer block output.
    Hook registers on model.transformer.ln_f and fires after the final LayerNorm,
    matching the post-ln_f space used to compute centroids.
    Returns softmax probabilities of shape (50257,).
    """
    inputs     = tokenizer(prompt, return_tensors="pt")
    dir_tensor = torch.tensor(direction, dtype=torch.float32)

    def hook_fn(module, input, output):
        # output[0] shape: (batch=1, seq_len, 768)
        modified = output[0] + alpha * dir_tensor.unsqueeze(0).unsqueeze(0)
        return (modified,) + output[1:]

    handle = model.transformer.ln_f.register_forward_hook(hook_fn)
    try:
        with torch.no_grad():
            outputs = model(**inputs)
        probs = torch.softmax(outputs.logits[0, -1, :], dim=-1).numpy()
    finally:
        handle.remove()
    return probs

# Sanity check: alpha=0 must reproduce get_probs() exactly
test_prompt = "On Monday I usually"
max_diff = np.abs(get_steered_probs(test_prompt, direction_norm, 0.0) - get_probs(test_prompt)).max()
print(f"Hook sanity check — max prob diff at alpha=0: {max_diff:.2e}  (should be < 1e-5)")

In [ ]:
alphas      = [0.0, 1.0, 2.0, 4.0]
demo_prompt = "On Monday I usually"

steered    = {a: get_steered_probs(demo_prompt, direction_norm, a) for a in alphas}
top_k      = 20
top_ids    = np.argsort(steered[0.0])[::-1][:top_k]
top_tokens = [tokenizer.decode([i]).strip() for i in top_ids]

fig, axes = plt.subplots(len(alphas), 1, figsize=(12, 9), sharex=True)
for ax, a in zip(axes, alphas):
    vals   = [steered[a][i] for i in top_ids]
    colors = ["steelblue" if steered[a][top_ids[k]] >= steered[0.0][top_ids[k]] else "coral"
              for k in range(top_k)]
    ax.bar(range(top_k), vals, color=colors, alpha=0.85)
    ax.set_ylabel(f"alpha={a}", fontsize=10)
    ax.set_ylim(0, max(steered[0.0][top_ids]) * 1.4)

axes[-1].set_xticks(range(top_k))
axes[-1].set_xticklabels(top_tokens, rotation=45, ha="right", fontsize=9)
axes[0].set_title(f'Token probability shift under Mon->Tue steering\nPrompt: "{demo_prompt}"', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
alphas_traj   = [0.0, 1.0, 2.0, 4.0]
monday_prompt = "On Monday I usually"
friday_prompt = "On Friday I usually"

def steered_beh_2d(prompt: str, direction: np.ndarray, alpha: float) -> np.ndarray:
    """Steer prompt, apply Hellinger embedding, project to 2D behavior PCA. Returns (2,)."""
    probs = get_steered_probs(prompt, direction, alpha)
    hell  = np.sqrt(probs)
    return pca_beh_2d.transform(hell.reshape(1, -1))[0]

mon_traj = np.array([steered_beh_2d(monday_prompt, direction_norm, a) for a in alphas_traj])
fri_traj = np.array([steered_beh_2d(friday_prompt, direction_norm, a) for a in alphas_traj])

fig, ax = plt.subplots(figsize=(8, 7))
for day in DAYS:
    ax.scatter(*beh_centroids_2d[day], color=colors_map[day], s=150, zorder=3,
               edgecolors="white", linewidths=1.5)
    ax.annotate(short[day], np.array(beh_centroids_2d[day]) + 0.002, fontsize=9)
for i in range(len(DAYS)):
    j = (i + 1) % len(DAYS)
    c1, c2 = beh_centroids_2d[DAYS[i]], beh_centroids_2d[DAYS[j]]
    ax.plot([c1[0], c2[0]], [c1[1], c2[1]], color="gray", lw=1, alpha=0.3)

ax.plot(mon_traj[:, 0], mon_traj[:, 1], "o-", color="steelblue", lw=2.5, ms=10,
        zorder=5, label="Mon prompt, steered Mon->Tue")
for i, a in enumerate(alphas_traj):
    ax.annotate(f"a={a}", mon_traj[i] + np.array([0.001, 0.002]), fontsize=8, color="steelblue")

ax.plot(fri_traj[:, 0], fri_traj[:, 1], "s--", color="red", lw=2.5, ms=10,
        zorder=5, label="Fri prompt, steered Mon->Tue (stress test)")
for i, a in enumerate(alphas_traj):
    ax.annotate(f"a={a}", fri_traj[i] + np.array([0.001, -0.002]), fontsize=8, color="red")

ax.set_title("Behavior Space Trajectory Under Steering\nBlue = cycle-local, Red = cross-cycle stress test", fontsize=11)
ax.set_xlabel("Behavior PC 1"); ax.set_ylabel("Behavior PC 2")
ax.legend(fontsize=9, loc="best")
plt.tight_layout()
plt.show()

mon_disp = np.linalg.norm(mon_traj[-1] - mon_traj[0])
fri_disp = np.linalg.norm(fri_traj[-1] - fri_traj[0])
print(f"Mon prompt — behavior displacement (alpha 0->4): {mon_disp:.4f}")
print(f"Fri prompt — behavior displacement (alpha 0->4): {fri_disp:.4f}")
print()
print("Note: single-prompt trajectories — treat as a heuristic stress test, not a proof.")
print("Similar displacements: suggests steering transfers smoothly across the boundary.")
print("Different displacements: consistent with Max's 'crack' — isometry may be local.")